# Objects and entities: one article, two modalities
### A hands-on teaching sample using the Wikipedia article *SK Sturm Graz*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SvenjaGuhr/SoSe2027_Computational_Methods_for_Digital_Humanities/blob/main/notebooks/ner_object_detection.ipynb)

We work with **one Wikipedia article in two ways**:

1. **Images → object detection.** We start here. We find objects in a picture from the article and see what the model can and cannot recognize.
2. **Text → named entity recognition.** We then find persons (**PER**) and organizations (**ORG**) in the running text, with two different models.

Both tasks share one idea: a model can only label the categories its **annotation scheme** defines. By the end you will be able to (1) run an object detector and read its boxes, (2) run a named entity recognizer and score it against a gold standard with precision, recall, and F1, and (3) see what changes when you are allowed to define the categories yourself.

*How to run:* click a cell and press **Shift+Enter**, from top to bottom.


## Step 0 — Setup
This installs spaCy, its small English model, and GLiNER. On Google Colab, `torch` and `torchvision` are already available. This takes a minute or two.

In [ ]:
# Install the libraries we need
!pip install -q spacy gliner
!python -m spacy download en_core_web_sm -q
print("Setup done. Please continue with the next cell.")

# Part A — Objects in images

Let us begin somewhere unexpected for a text-focused field: a photograph. The question is the same one we will later ask of text. What are the "things" here, and how does a model find them?

We use a detector trained on **COCO**, a dataset with 80 everyday object classes. The detector draws a **bounding box** around each object it recognizes and labels it.

First, give the notebook an image. Run the cell and upload a photo from the article, for example a team photo or a match photo.

*Rights reminder:* the article text is CC BY-SA, but each image carries its own license. Photos on Wikimedia Commons are often freely licensed, while the club crest is usually a non-free logo. For class display this is fine; for anything you publish, check each image's license.

In [ ]:
from google.colab import files

uploaded = files.upload()        # choose one image file
image_path = next(iter(uploaded))
print("Using image:", image_path)

Now run the detector. The first run downloads the model weights (about 160 MB), which takes a moment.

In [ ]:
import torch
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
)
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

COCO_CATEGORIES = [
    "__background__","person","bicycle","car","motorcycle","airplane","bus","train",
    "truck","boat","traffic light","fire hydrant","N/A","stop sign","parking meter",
    "bench","bird","cat","dog","horse","sheep","cow","elephant","bear","zebra",
    "giraffe","N/A","backpack","umbrella","N/A","N/A","handbag","tie","suitcase",
    "frisbee","skis","snowboard","sports ball","kite","baseball bat","baseball glove",
    "skateboard","surfboard","tennis racket","bottle","N/A","wine glass","cup","fork",
    "knife","spoon","bowl","banana","apple","sandwich","orange","broccoli","carrot",
    "hot dog","pizza","donut","cake","chair","couch","potted plant","bed","N/A",
    "dining table","N/A","N/A","toilet","N/A","tv","laptop","mouse","remote","keyboard",
    "cell phone","microwave","oven","toaster","sink","refrigerator","N/A","book","clock",
    "vase","scissors","teddy bear","hair drier","toothbrush",
]

THRESHOLD = 0.6   # only show detections the model is at least 60% sure about

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()

image = Image.open(image_path).convert("RGB")
tensor = weights.transforms()(image)
with torch.no_grad():
    prediction = model([tensor])[0]

draw = ImageDraw.Draw(image)
try:
    font = ImageFont.truetype("DejaVuSans.ttf", 18)
except OSError:
    font = ImageFont.load_default()

print(f"Detections (score >= {THRESHOLD}):")
found = 0
for box, label_id, score in zip(prediction["boxes"], prediction["labels"], prediction["scores"]):
    if score < THRESHOLD:
        continue
    found += 1
    label = COCO_CATEGORIES[label_id]
    x0, y0, x1, y1 = [float(v) for v in box]
    print(f"   {label:<14} {score:.2f}")
    draw.rectangle([x0, y0, x1, y1], outline="red", width=3)
    draw.text((x0 + 2, y0 + 2), f"{label} {score:.2f}", fill="red", font=font)
if found == 0:
    print("   (nothing above the threshold - try lowering THRESHOLD)")

plt.figure(figsize=(10, 8))
plt.imshow(image)
plt.axis("off")
plt.show()

### What the detector can and cannot see

Every label above comes from the 80-class COCO scheme. On a football photo you will typically see `person` and `sports ball`, and little else. There is **no class for a club crest, a jersey, or a goal**, so the model cannot find them, even though they are the most football-specific things in the picture.

That gap is the whole point: the model sees only the categories its scheme defines. Keep this in mind as we turn to text, where the same thing happens with persons and organizations.

**To take further:** lower `THRESHOLD` to 0.3 and re-run. More boxes appear, but more of them are wrong. That trade-off between finding more (recall) and being right (precision) is exactly what we measure next.

# Part B — Named entities in text

A **named entity** is a span of text that names a thing of a certain type. We care about two:
- **PER** — a person (a coach or a player)
- **ORG** — an organization (a football club or a league)

We start with the trained model `en_core_web_sm`. Note that spaCy calls the person type `PERSON`; we treat that as `PER`.

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

# A short teaching passage built from well-established facts about the club.
# You can also paste the real article text here later.
passage = (
    "SK Sturm Graz is an Austrian association football club based in Graz. "
    "The club plays in the Austrian Bundesliga and has represented Austria in "
    "the UEFA Champions League and the UEFA Europa League. "
    "Sturm's main rival is Grazer AK, and meetings between the two are called the Graz derby. "
    "The club has also faced Rapid Wien and Austria Wien in league competition. "
    "Sturm play their home games in the Liebenau district of Graz, at a ground "
    "once known as the Arnold Schwarzenegger Stadion. "
    "Among its best-known coaches are Ivica Osim and Franco Foda; "
    "Foda later managed the Austria national team."
)

doc = nlp(passage)

# Interactive, color-coded view of every entity the model found:
displacy.render(doc, style="ent", jupyter=True)

### What did the model find?
The table lists every entity, its label, and what the label means. Look closely at how the football clubs are labeled.

In [ ]:
import pandas as pd

rows = [(ent.text, ent.label_, spacy.explain(ent.label_)) for ent in doc.ents]
pd.DataFrame(rows, columns=["text", "spaCy label", "meaning"])

### Compare with a gold standard

A **gold standard** is the set of correct answers a human annotator agrees on. We compare the model's answers to it and compute:
- **Precision** — of the spans the model proposed, how many were correct.
- **Recall** — of the spans that should have been found, how many it found.
- **F1** — the balance of the two.

In [ ]:
gold = {
    ("SK Sturm Graz", "ORG"), ("Austrian Bundesliga", "ORG"),
    ("UEFA Champions League", "ORG"), ("UEFA Europa League", "ORG"),
    ("Sturm", "ORG"), ("Grazer AK", "ORG"), ("Rapid Wien", "ORG"),
    ("Austria Wien", "ORG"), ("Austria national team", "ORG"),
    ("Arnold Schwarzenegger", "PER"), ("Ivica Osim", "PER"), ("Franco Foda", "PER"),
}

def norm(label):
    return "PER" if label == "PERSON" else label

pred = {(ent.text, norm(ent.label_)) for ent in doc.ents if norm(ent.label_) in ("PER", "ORG")}

true_pos = sorted(pred & gold)
false_pos = sorted(pred - gold)   # model proposed, but wrong
false_neg = sorted(gold - pred)   # should have found, but missed

print("CORRECT (model agrees with gold):")
for text, label in true_pos:
    print(f"   {label:<4} {text}")
print("\nWRONG or EXTRA (model proposed, not in gold):")
for text, label in false_pos:
    print(f"   {label:<4} {text}")
print("\nMISSED (in gold, model did not find):")
for text, label in false_neg:
    print(f"   {label:<4} {text}")

P = len(true_pos) / (len(true_pos) + len(false_pos)) if pred else 0
R = len(true_pos) / (len(true_pos) + len(false_neg)) if gold else 0
F1 = 2 * P * R / (P + R) if (P + R) else 0
print(f"\nPrecision {P:.2f}   Recall {R:.2f}   F1 {F1:.2f}")

### What the errors tell us

The score is low, and the mistakes are the lesson:

1. **Clubs read as people.** "Sturm", "Grazer AK", "Rapid Wien", "Austria Wien", and even "Bundesliga" are tagged `PERSON`. A general English model has rarely seen Austrian club names, so it guesses that an unfamiliar capitalized name is a person.
2. **Boundary errors.** "SK Sturm Graz" comes back without the "SK"; "Austrian Bundesliga" is split in two.
3. **An embedded person goes missing.** Because the whole "Arnold Schwarzenegger Stadion" is taken as one span, the person inside it is never found.
4. **The same word, different labels.** "Graz" is a place in one sentence and something else in another.
5. **Inconsistency.** "UEFA Champions League" is correct, while "UEFA Europa League" right after it is missed.

The fixed scheme is the root cause: `en_core_web_sm` only knows its built-in types, and "football club" is not one of them, so clubs are forced into the nearest familiar box, `PERSON`.

### A model you can give your own labels: GLiNER

What if we could **define the categories ourselves**? GLiNER is an open named entity recognizer built on a bidirectional transformer. It is not a generative large language model system; instead, it takes a list of labels you provide at prediction time and finds spans that fit them, without being retrained. This lets us name "football club" directly and see whether the clubs are recovered.

The model downloads on first use. `gliner_small-v2.1` is fast and English-focused; for multilingual text, including German, swap in `urchade/gliner_multi-v2.1`.

In [ ]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

# You define the label set. These are not built in; you are writing the scheme.
labels = ["person", "football club", "league or competition", "stadium", "city"]

entities = gliner_model.predict_entities(passage, labels, threshold=0.5)
for ent in entities:
    print(f"   {ent['label']:<22} {ent['text']}   ({ent['score']:.2f})")

### What GLiNER shows

Because you supplied "football club" as a label, GLiNER can place the Austrian clubs in a category that fits them, where `en_core_web_sm` had no choice but `PERSON`. You can also split organizations into finer types, such as "league or competition" and "stadium", simply by naming them.

This is the clearest statement of the lesson: **the categories live in the scheme, and here you write the scheme**. The same move that let COCO see only its 80 object classes, and let spaCy see only its built-in entity types, is now in your hands. Try changing the `labels` list, for example adding "country" or "nationality", and re-run to see how the output shifts.

A word of care: a freely chosen label set is powerful and also fragile. Vague or overlapping labels produce inconsistent spans, so the discipline of writing clear annotation guidelines matters as much here as in manual annotation.

### Try it yourself

Replace the text below with your own sentence or a paragraph from another article, then run it through both models.

In [ ]:
your_text = "Sturm Graz beat Rapid Wien at the Merkur Arena, and coach Jürgen Säumel was pleased."

print("spaCy:")
displacy.render(nlp(your_text), style="ent", jupyter=True)

print("\nGLiNER:")
for ent in gliner_model.predict_entities(your_text, labels, threshold=0.5):
    print(f"   {ent['label']:<22} {ent['text']}   ({ent['score']:.2f})")

# Wrap-up

One Wikipedia article, two modalities, one lesson:

- Categories live in the **annotation scheme**. COCO sees 80 object classes, `en_core_web_sm` sees its built-in entity types, and GLiNER sees whatever labels you give it.
- **Evaluation needs a gold standard**, with precision, recall, and F1, for text and for images alike.
- Freedom to define categories still rests on clear **guidelines**, which is how we keep annotation intersubjective.

When you choose data for your group project, remember that the same pipeline of annotate, automate, and evaluate works for text, images, and audio.